# Grafema AI · Evaluación de búsqueda híbrida

Compara **TF-IDF**, **BGE-M3** y una búsqueda híbrida que combina sus puntajes con pesos **fijos de 50 % / 50 %**. Usa las mismas 23 consultas y los mismos 332 cantos etiquetados por el índice temático del PDF.

**Método:** se obtienen similitudes coseno para BGE-M3 y puntajes TF-IDF con trigramas a pentagramas de caracteres. Se normaliza el rango de puntajes de **cada consulta** a 0–1 sobre los mismos 332 cantos y se usa `0.5 × TF-IDF + 0.5 × BGE-M3`. También se muestran 25/75 y 75/25 como análisis exploratorio; no se elige un peso ganador usando el mismo benchmark. Las categorías del PDF solo intervienen al evaluar, nunca se incorporan a los textos ni a los embeddings.

El modelo puede devolver vectores ligeramente distintos en otra ejecución: el notebook **recalcula la línea base semántica** con los mismos vectores de consulta usados por la combinación y muestra cuántos resultados del top 10 coinciden con la primera evaluación. Por eso la comparación de los tres métodos sigue siendo válida aunque el orden semántico difiera del informe anterior.


In [ ]:
from pathlib import Path
import json, hashlib

needed = ["songs_clean.json", "benchmark_tematico_pdf.json", "embeddings_bge_m3.json", "evaluacion_busqueda.json"]
missing = [name for name in needed if not Path(name).is_file()]
if missing:
    try:
        from google.colab import files
        print("Sube estos archivos:", ", ".join(missing))
        files.upload()
    except ImportError:
        print("Coloca estos archivos junto al notebook:", ", ".join(missing))
for name in needed:
    assert Path(name).is_file(), f"Falta {name}"

catalog=json.loads(Path(needed[0]).read_text(encoding="utf-8"))
benchmark=json.loads(Path(needed[1]).read_text(encoding="utf-8"))
embeddings=json.loads(Path(needed[2]).read_text(encoding="utf-8"))
previous=json.loads(Path(needed[3]).read_text(encoding="utf-8"))
songs=catalog["songs"]
queries=benchmark["queries"]
ids={song["songId"] for category in benchmark["categories"] for song in category["songs"]}
ordered_ids=[song["id"] for song in songs if song["id"] in ids]
by_id={song["id"]:song for song in songs}
texts={song["id"]:f"Título: {song['title']}\nLetra: {song['lyrics']}" for song in songs}
fingerprint=hashlib.sha256(json.dumps([(song["id"],texts[song["id"]]) for song in songs],ensure_ascii=False,separators=(",",":")).encode("utf-8")).hexdigest()
assert len(songs)==len(by_id)==len(embeddings["vectors"])==550
assert len(ordered_ids)==len(ids)==332 and len(queries)==23
assert set(by_id)==set(embeddings["vectors"])
assert fingerprint==embeddings["fingerprint"]==previous["catalogFingerprint"]
assert embeddings["model"]==previous["model"]=="@cf/baai/bge-m3"
assert [q["id"] for q in queries]==[q["queryId"] for q in previous["methods"][0]["queries"]]
print("Datos verificados: 550 cantos, 332 evaluables y 23 consultas.")

Sube estos archivos: songs_clean.json, benchmark_tematico_pdf.json, embeddings_bge_m3.json, evaluacion_busqueda.json


Saving embeddings_bge_m3.json to embeddings_bge_m3.json
Saving evaluacion_busqueda.json to evaluacion_busqueda.json
Saving benchmark_tematico_pdf.json to benchmark_tematico_pdf.json
Saving songs_clean.json to songs_clean.json
Datos verificados: 550 cantos, 332 evaluables y 23 consultas.


In [ ]:
# Si el notebook previo sigue abierto en este mismo entorno, query_vectors ya existe.
# Si existe el archivo de consultas de una ejecución previa, se reutiliza sin llamadas a Cloudflare.
query_file=Path("query_embeddings_bge_m3.json")
if query_file.is_file():
    cached=json.loads(query_file.read_text(encoding="utf-8"))
    assert cached["model"]==embeddings["model"]
    assert cached["queries"]==[{"id":q["id"],"query":q["query"]} for q in queries]
    query_vectors=cached["vectors"]
elif "query_vectors" in globals() and isinstance(query_vectors,list) and len(query_vectors)==len(queries):
    print("Se usarán los vectores de consultas que siguen en memoria del notebook anterior.")
else:
    from getpass import getpass
    from urllib.parse import quote
    import requests, time
    account_id=input("Cloudflare Account ID: ").strip()
    api_token=getpass("Workers AI API Token (no se mostrará): ").strip()
    assert account_id and api_token,"Faltan credenciales"
    url=f"https://api.cloudflare.com/client/v4/accounts/{quote(account_id,safe='')}/ai/run/{embeddings['model']}"
    session=requests.Session()
    session.headers.update({"Authorization":f"Bearer {api_token}"})
    def embed_queries(batch):
        for attempt in range(6):
            try:
                response=session.post(url,json={"text":batch},timeout=120)
            except requests.RequestException:
                if attempt==5: raise
                time.sleep(min(2**attempt,20))
                continue
            if response.status_code in (429,500,502,503,504) and attempt<5:
                time.sleep(min(2**attempt+1,30))
                continue
            if not response.ok: raise RuntimeError(f"Cloudflare HTTP {response.status_code}: {response.text[:500]}")
            body=response.json()
            if body.get("success") is not True: raise RuntimeError(f"Cloudflare rechazó la solicitud: {body.get('errors')}")
            vectors=body.get("result",{}).get("data")
            if not isinstance(vectors,list) or len(vectors)!=len(batch): raise ValueError("La respuesta de embeddings no coincide con la cantidad de consultas")
            return vectors
        raise RuntimeError("Se agotaron los reintentos")
    query_vectors=[]
    for start in range(0,len(queries),8):
        query_vectors.extend(embed_queries([q["query"] for q in queries[start:start+8]]))
    session.close()
    del api_token, session

assert len(query_vectors)==len(queries)==23
assert all(isinstance(v,list) and len(v)==len(next(iter(embeddings["vectors"].values()))) for v in query_vectors)
query_file.write_text(json.dumps({"model":embeddings["model"],"queries":[{"id":q["id"],"query":q["query"]} for q in queries],"vectors":query_vectors},ensure_ascii=False)+"\n",encoding="utf-8")
print("Vectores de consultas listos:",len(query_vectors))

Cloudflare Account ID: 932465ed511d0a8553dde4e31bac09e8
Workers AI API Token (no se mostrará): ··········
Vectores de consultas listos: 23


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# Exactamente el mismo TF-IDF del benchmark anterior.
tfidf=TfidfVectorizer(strip_accents="unicode",analyzer="char_wb",ngram_range=(3,5),sublinear_tf=True)
doc_matrix=tfidf.fit_transform([texts[song_id] for song_id in ordered_ids])
lexical_scores=(tfidf.transform([q["query"] for q in queries])@doc_matrix.T).toarray()

def normalize_rows(values):
    matrix=np.asarray(values,dtype=np.float32)
    norms=np.linalg.norm(matrix,axis=1,keepdims=True)
    if not np.all(np.isfinite(matrix)) or np.any(norms==0): raise ValueError("Embedding inválido")
    return matrix/norms

semantic_scores=normalize_rows(query_vectors)@normalize_rows([embeddings["vectors"][song_id] for song_id in ordered_ids]).T
assert lexical_scores.shape==semantic_scores.shape==(23,332)

# TF-IDF es determinista y sirve para detectar cambios de orden en el catálogo.
# Los vectores de consulta de BGE-M3 pueden cambiar entre llamadas: se compara
# su top 10, pero la línea base semántica se recalcula con ESTA ejecución.
old_lexical=next(m for m in previous["methods"] if "TF-IDF" in m["method"])
old_semantic=next(m for m in previous["methods"] if "TF-IDF" not in m["method"])
semantic_overlap=[]
for i,(lex_row,sem_row) in enumerate(zip(old_lexical["queries"],old_semantic["queries"])):
    lex_top=[ordered_ids[j] for j in np.argsort(-lexical_scores[i],kind="stable")[:10]]
    assert lex_top==[item["songId"] for item in lex_row["top10"]],f"TF-IDF no reproduce {lex_row['queryId']}"
    sem_top=[ordered_ids[j] for j in np.argsort(-semantic_scores[i],kind="stable")[:10]]
    old_top=[item["songId"] for item in sem_row["top10"]]
    semantic_overlap.append({"queryId":sem_row["queryId"],"commonTop10":len(set(sem_top)&set(old_top)),
                             "sameOrder":sem_top==old_top})
changed=[item for item in semantic_overlap if not item["sameOrder"]]
print("TF-IDF reproduce los 23 top-10 previos.")
if changed:
    print(f"BGE-M3 difiere en {len(changed)} de 23 consultas. Se recalculará su línea base con los vectores actuales.")
    print("Consultas distintas (coincidencias sin considerar el orden):",[(x["queryId"],x["commonTop10"]) for x in changed])
else:
    print("BGE-M3 también reproduce los 23 top-10 previos.")

def scale_per_query(scores):
    low=scores.min(axis=1,keepdims=True)
    high=scores.max(axis=1,keepdims=True)
    diff=high-low
    if np.any(diff<=0): raise ValueError("Todos los cantos recibieron el mismo puntaje en una consulta")
    return (scores-low)/diff

lex01=scale_per_query(lexical_scores)
sem01=scale_per_query(semantic_scores)
hybrid_scores=0.5*lex01+0.5*sem01
assert np.isfinite(hybrid_scores).all()
print("Fusión 50/50 calculada para los 332 cantos en cada consulta.")

TF-IDF reproduce los 23 top-10 previos.
BGE-M3 difiere en 2 de 23 consultas. Se recalculará su línea base con los vectores actuales.
Consultas distintas (coincidencias sin considerar el orden): [('q10', 10), ('q22', 9)]
Fusión 50/50 calculada para los 332 cantos en cada consulta.


In [ ]:
# Conserva el formato del primer informe para mostrarlo en el dashboard.
def evaluate(scores,method):
    rows=[]
    for i,query in enumerate(queries):
        positive=set(query["positiveSongIds"])
        ranking=np.argsort(-scores[i],kind="stable")
        top10=[ordered_ids[j] for j in ranking[:10]]
        rows.append({"queryId":query["id"],"query":query["query"],"categoryId":query["categoryId"],
                     "positivesInPdf":len(positive),"hitAt5":int(bool(positive.intersection(top10[:5]))),
                     "recallOfListedPositivesAt10":round(len(positive.intersection(top10))/len(positive),4),
                     "top10":[{"songId":song_id,"title":by_id[song_id]["title"],
                               "score":round(float(scores[i,ranking[rank]]),4),
                               "listedInCategory":song_id in positive} for rank,song_id in enumerate(top10)]})
    return {"method":method,"meanHitAt5":round(float(np.mean([row["hitAt5"] for row in rows])),4),
            "meanListedRecallAt10":round(float(np.mean([row["recallOfListedPositivesAt10"] for row in rows])),4),
            "queries":rows}

lexical=evaluate(lexical_scores,"TF-IDF char 3-5")
semantic=evaluate(semantic_scores,"bge-m3 cosine")
hybrid=evaluate(hybrid_scores,"Híbrido 50/50 (min-max por consulta)")
methods=[lexical,semantic,hybrid]
exploratory=[]
for w in (0.25,0.75):
    result=evaluate((1-w)*lex01+w*sem01,f"Híbrido {round((1-w)*100)}/{round(w*100)} (TF-IDF/BGE-M3)")
    exploratory.append({"weightSemantic":w,"meanHitAt5":result["meanHitAt5"],"meanListedRecallAt10":result["meanListedRecallAt10"]})
print(f"{'Método':<38} {'Hit@5':>8} {'Recall@10':>11}")
for result in methods:
    print(f"{result['method']:<38} {result['meanHitAt5']:>8.4f} {result['meanListedRecallAt10']:>11.4f}")
print("Pesos exploratorios (no seleccionados según el benchmark):",exploratory)
report={"sourceBenchmark":"benchmark_tematico_pdf.json","catalogSize":len(songs),"evaluationScope":len(ordered_ids),
        "queryCount":len(queries),"model":embeddings["model"],"catalogFingerprint":fingerprint,
        "metricsNote":"Evaluación restringida a 332 cantos del PDF; los demás no se tratan como negativos. Resultados sobre 23 consultas, sin separar entrenamiento y prueba.",
        "hybridDescription":"TF-IDF char 3-5 y similitud coseno BGE-M3 escalados min-max por consulta sobre los mismos 332 documentos; media 50/50 fijada antes de evaluar.",
        "baselineComparison":{"semanticTop10OverlapWithPrevious":semantic_overlap,
                              "changedSemanticQueries":len(changed),
                              "note":"BGE-M3 se recalcula con los mismos vectores de consulta usados por el híbrido; no se mezclan puntuaciones de ejecuciones distintas."},
        "methods":methods,"exploratoryWeights":exploratory}
Path("evaluacion_hibrida.json").write_text(json.dumps(report,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
print("Guardado: evaluacion_hibrida.json")
try:
    from google.colab import files
    files.download("evaluacion_hibrida.json")
    files.download("query_embeddings_bge_m3.json")
except ImportError:
    print("Archivos guardados en el directorio de trabajo.")

Método                                    Hit@5   Recall@10
TF-IDF char 3-5                          0.6957      0.2211
bge-m3 cosine                            0.7391      0.2926
Híbrido 50/50 (min-max por consulta)     0.7826      0.2492
Pesos exploratorios (no seleccionados según el benchmark): [{'weightSemantic': 0.25, 'meanHitAt5': 0.6957, 'meanListedRecallAt10': 0.255}, {'weightSemantic': 0.75, 'meanHitAt5': 0.7826, 'meanListedRecallAt10': 0.2872}]
Guardado: evaluacion_hibrida.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>